# 05_high_spend_by_tier

Generated from `alteryx-workflows\05_high_spend_by_tier.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `orders`, `customers`.


In [ ]:
orders = spark.createDataFrame([["C001", "West", "Online", "2026-01-12", "SKU-A-100", "1200", "4"], ["C002", "East", "Store", "2026-01-18", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "2026-02-03", "SKU-A-140", "640", "1"], ["C003", "North", "Partner", "2026-02-11", "SKU-C-010", "2100", "6"], ["C004", "South", "Store", "2026-03-01", "SKU-B-220", "430", "3"], ["C002", "East", "Online", "2026-03-09", "SKU-A-100", "990", "2"], ["C005", "West", "Partner", "2026-03-21", "SKU-C-010", "150", "1"], ["C003", "North", "Store", "2026-04-02", "SKU-A-140", "1760", "5"]], ["CustomerID", "Region", "Channel", "OrderDate", "SKU", "Amount", "Units"])
customers = spark.createDataFrame([["C001", "Acme Retail", "Gold"], ["C002", "Northwind", "Silver"], ["C003", "Contoso", "Gold"], ["C004", "Fabrikam", "Bronze"], ["C005", "Adventure", "Silver"]], ["CustomerID", "CustomerName", "Tier"])


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 1

Produces `amount_number`, `units_number`, `orders_to_customers`, `spend_by_tier`, and 1 more.


In [ ]:
amount_number = orders.withColumn("AmountNum", F.col("Amount").cast('double'))
units_number = amount_number.withColumn("UnitsNum", F.col("Units").cast('double'))
orders_to_customers = units_number.join(customers, (units_number["CustomerID"] == customers["CustomerID"]), 'inner').drop(customers["CustomerID"])
spend_by_tier = orders_to_customers.groupBy(*["Tier"]).agg(F.sum(F.col("AmountNum").cast('double')).alias("Sum_AmountNum"), F.count(F.col("AmountNum")).alias("RowCount"))
rank_tiers = spend_by_tier.orderBy(F.col("Sum_AmountNum").desc())


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(rank_tiers)
display(orders_to_customers)
